# 🏠 Kaggle: House Prices — Full Lab Notebook

**Student:** Biswas Khatiwada  
**Course:** Kaggle Lab — House Prices: Advanced Regression Techniques  
**Competition:** https://www.kaggle.com/c/house-prices-advanced-regression-techniques

---
### What this notebook does
- Loads the competition data from `/kaggle/input/house-prices-advanced-regression-techniques/`
- EDA: quick visuals + missingness overview
- Preprocessing & Feature Engineering (numeric/categorical pipelines)
- Trains and compares multiple models (with RMSE):
  - Linear Regression
  - Random Forest
  - Gradient Boosting
  - XGBoost *(if available)*
  - LightGBM *(if available)*
  - CatBoost *(if available)*
- Plots Top-10 feature importances for the best model (when supported)
- Generates `submission.csv` for Kaggle upload

---
### Required Artifacts for the Lab
- **Kaggle submission screenshot** (score + rank)
- **Public notebook URL** (share link)
- **Reflection** (what worked, what to improve)

> **Tip:** If a library (e.g., LightGBM/CatBoost) isn't available, this notebook will skip it gracefully and still submit with the best available model.

## 1) Set-up & Imports
- Detect optional libraries
- Define helpers (RMSE, plotting, model runner)

In [ ]:
# ===============================
# 0) Imports & utility
# ===============================
import os
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_squared_error
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

def rmse(y_true, y_pred):
    """Root Mean Squared Error metric"""
    return mean_squared_error(y_true, y_pred, squared=False)


 Data path setup

In [ ]:
# ===============================
# 1) Data paths (works both on Kaggle & local)
# ===============================
KAGGLE_BASE = "/kaggle/input/house-prices-advanced-regression-techniques"

if os.path.exists(KAGGLE_BASE):
    TRAIN_PATH = os.path.join(KAGGLE_BASE, "train.csv")
    TEST_PATH  = os.path.join(KAGGLE_BASE, "test.csv")
else:
    TRAIN_PATH = "train.csv"
    TEST_PATH  = "test.csv"


## 2) Load Data
Kaggle places the files at `/kaggle/input/house-prices-advanced-regression-techniques/`.


In [ ]:
# ===============================
# 2) Load data
# ===============================
train_df = pd.read_csv(TRAIN_PATH)
test_df  = pd.read_csv(TEST_PATH)

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)
train_df.head()


Split features and target


In [ ]:
# ===============================
# 3) Define features and target
# ===============================
X = train_df.drop(columns=["SalePrice", "Id"])
y = np.log1p(train_df["SalePrice"])  # log-transform for stability
X_test = test_df.drop(columns=["Id"])


Train-validation split

In [ ]:
# ===============================
# 4) Train-validation split
# ===============================
X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Train size:", X_train.shape, "Validation size:", X_valid.shape)


Preprocessing pipeline

In [ ]:
# ===============================
# 5) Define preprocessing (dense output)
# ===============================
numeric_features = X_train.select_dtypes(include=[np.number]).columns
categorical_features = X_train.select_dtypes(exclude=[np.number]).columns

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# Handle sklearn versions:
# - 1.2+ uses 'sparse_output'
# - older uses 'sparse'
try:
    categorical_transformer = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
    ])
except TypeError:
    categorical_transformer = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", sparse=False))
    ])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ],
    remainder="drop"
)

print(f"Numeric features: {len(numeric_features)} | Categorical features: {len(categorical_features)}")



Candidate models

In [ ]:
# ===============================
# 6) Candidate models (Ridge uses lsqr to avoid cg)
# ===============================
candidates = [
    ("LinearRegression", LinearRegression()),
    ("Ridge", Ridge(alpha=1.0, random_state=42, solver="lsqr")),  # <- changed
    ("Lasso", Lasso(alpha=0.0005, random_state=42, max_iter=20000)),
    ("RandomForest", RandomForestRegressor(n_estimators=500, random_state=42, n_jobs=-1)),
    ("GradientBoosting", GradientBoostingRegressor(random_state=42))
]


Model training & evaluation

In [ ]:
# ===============================
# 7) Train, validate, and compare
# ===============================
LOG_MIN, LOG_MAX = 0.0, 20.0
results = []
fitted_models = {}

for name, model in candidates:
    pipe = Pipeline(steps=[("prep", preprocessor), ("model", model)])
    pipe.fit(X_train, y_train)

    y_log_pred = pipe.predict(X_valid)
    y_log_pred = np.clip(y_log_pred, LOG_MIN, LOG_MAX)
    preds_valid = np.expm1(y_log_pred)
    score = rmse(np.expm1(y_valid), preds_valid)

    results.append((name, score))
    fitted_models[name] = pipe

results_df = pd.DataFrame(results, columns=["Model", "RMSE"]).sort_values("RMSE")
print("Validation RMSE (lower is better):")
print(results_df.to_string(index=False))



Pick best model

In [ ]:
# ===============================
# 8) Pick best model
# ===============================
best_name = results_df.iloc[0, 0]
best_pipe = fitted_models[best_name]

print(f"✅ Best model: {best_name}")


Retrain on full dataset

In [ ]:
# ===============================
# 9) Retrain best model on all data
# ===============================
best_pipe.fit(X, y)


Predict test set

In [ ]:
# ===============================
# 10) Predict test set and generate submission
# ===============================
test_log_pred = best_pipe.predict(X_test)
test_log_pred = np.clip(test_log_pred, LOG_MIN, LOG_MAX)
test_price_pred = np.expm1(test_log_pred)

submission = pd.DataFrame({
    "Id": test_df["Id"],
    "SalePrice": test_price_pred
})
submission.to_csv("submission.csv", index=False)
print("✅ Submission file created:", submission.shape)
submission.head()


Plot RMSE comparison

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8,4))
plt.bar(results_df["Model"], results_df["RMSE"])
plt.xticks(rotation=30, ha='right')
plt.ylabel("RMSE (Validation)")
plt.title("Model Performance Comparison")
plt.show()

